# Metric 0 — Translation Time
Run from project root: `jupyter notebook evaluation/visualisations/metric_0_timing.ipynb`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

METRICS_DIR = Path('../../eval_results/metrics')
PATH = METRICS_DIR / 'metric_0.jsonl'

CONFIGS = [
    'api_no_glos', 'api_glos',
    'local_no_glos', 'local_glos',
    'finetuned_no_glos', 'finetuned_glos',
]
LABELS = {
    'api_no_glos':       'API (no glos)',
    'api_glos':          'API (glos)',
    'local_no_glos':     'Local (no glos)',
    'local_glos':        'Local (glos)',
    'finetuned_no_glos': 'Finetuned (no glos)',
    'finetuned_glos':    'Finetuned (glos)',
}

rows = []
with open(PATH) as f:
    for line in f:
        rec = json.loads(line)
        row = {'idx': rec['idx']}
        for ck in CONFIGS:
            row[ck] = rec.get(ck)
        rows.append(row)

df = pd.DataFrame(rows).set_index('idx')
print(f'Loaded {len(df)} examples')
df.head()

In [ ]:
# --- Average time per config ---
avgs = df[CONFIGS].mean().dropna()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([LABELS[c] for c in avgs.index], avgs.values, color=sns.color_palette('muted', len(avgs)))
ax.bar_label(bars, fmt='%.1fs', padding=3)
ax.set_ylabel('Avg time per example (seconds)')
ax.set_title('Metric 0 — Average Translation Time per Config')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Box plot: time distribution per config ---
plot_data = {LABELS[c]: df[c].dropna().values for c in CONFIGS if c in df.columns and df[c].notna().any()}

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(plot_data.values(), labels=plot_data.keys(), patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Time (seconds)')
ax.set_title('Metric 0 — Translation Time Distribution')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Grouped bar: no_glos vs glos per model ---
models = ['api', 'local', 'finetuned']
model_labels = ['API', 'Local', 'Finetuned']
no_glos = [df[f'{m}_no_glos'].mean() if f'{m}_no_glos' in df else np.nan for m in models]
glos    = [df[f'{m}_glos'].mean()    if f'{m}_glos'    in df else np.nan for m in models]

x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, no_glos, w, label='No glossary', color='steelblue')
b2 = ax.bar(x + w/2, glos,    w, label='Glossary',    color='coral')
ax.bar_label(b1, fmt='%.1fs', padding=3)
ax.bar_label(b2, fmt='%.1fs', padding=3)
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel('Avg time (seconds)')
ax.set_title('Metric 0 — Glossary vs No Glossary Time per Model')
ax.legend()
plt.tight_layout()
plt.show()